# Polygroups of order 5 — verification notebook

Companion notebook of the paper *An Enumeration and Classification of Non-isomorphic Polygroups of Order 5* (Discussiones Mathematicae – General Algebra and Applications, revised version).

It reproduces, from the files of this repository, every computational statement of the revised paper:

1. **Step 1** – the 4,412 tables of `data/polygroups_order5.csv` are polygroups, pairwise non-isomorphic, and their stored keys are canonical.
2. **Step 2** – an *independent* exhaustive enumeration (orbits of the reversibility relation, `src/enum_indep.cpp`) gives the same 4,412 isomorphism classes, and the orders 2, 3, 4 give 2, 10, 102.
3. **Step 3** – all invariants and all tables of the paper (inverse types, primitive/non-primitive, fundamental group, commutativity, algebraic generation number, logical generation number $\lambda(P)$ via automorphisms).
4. **Step 4 (optional, 15–40 min)** – recompile and rerun the main two-phase search program `src/polygroups_n5.cpp`.

Run it on Google Colab with *Runtime → Run all*. Every check ends with an `assert`; if everything is fine the last cell prints `ALL CHECKS PASSED`.

In [1]:
import os, subprocess, sys, itertools, collections, re, time

REPO_URL = "https://github.com/Pedyi/Polygroups-Order-5-Enumeration"
if os.path.exists("data/polygroups_order5.csv"):
    ROOT = "."
elif os.path.exists("../data/polygroups_order5.csv"):
    ROOT = ".."
else:                                    # e.g. on Google Colab
    if not os.path.exists("repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    ROOT = "repo"
print("repository root:", os.path.abspath(ROOT))
print(subprocess.run(["g++", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

repository root: /home/claude/w/repo_new
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


In [2]:
# ---------- basic tools (elements 0..n-1, identity = 0; a table T[x][y] is a bit mask of the subset x o y) ----------
def parse_csv(path, n=5):
    tabs, cur = [], []
    for line in open(path, encoding="utf8"):
        line = line.strip()
        if line.startswith("Unique Structure"):
            cur = []
        elif line.startswith('"'):
            cells = re.findall(r"\{([0-9,]*)\}", line)
            cur.append([sum(1 << (int(v) - 1) for v in c.split(",") if v) for c in cells])
            if len(cur) == n:
                tabs.append(cur)
    return tabs

def inverse_map(T, n=5):
    inv = []
    for x in range(n):
        ys = [y for y in range(n) if T[x][y] & 1]
        if len(ys) != 1: return None
        inv.append(ys[0])
    return inv

def is_polygroup(T, n=5):
    inv = inverse_map(T, n)
    if inv is None: return False
    for x in range(n):
        if T[0][x] != 1 << x or T[x][0] != 1 << x: return False       # axiom (ii)
        if inv[inv[x]] != x: return False                             # Lemma 1(c)
        for y in range(n):
            if T[x][y] == 0: return False
    for x in range(n):
        for y in range(n):
            for z in range(n):
                if T[x][y] >> z & 1:                                  # axiom (iii)
                    if not (T[z][inv[y]] >> x & 1) or not (T[inv[x]][z] >> y & 1): return False
    for i in range(n):                                                # axiom (i)
        for j in range(n):
            for l in range(n):
                lhs = 0
                for s in range(n):
                    if T[i][j] >> s & 1: lhs |= T[s][l]
                rhs = 0
                for t in range(n):
                    if T[j][l] >> t & 1: rhs |= T[i][t]
                if lhs != rhs: return False
    return True

def relabel(T, p, n=5):          # (p.T)(p(x),p(y)) = p(T(x,y))
    R = [[0] * n for _ in range(n)]
    for x in range(n):
        for y in range(n):
            m = 0
            for z in range(n):
                if T[x][y] >> z & 1: m |= 1 << p[z]
            R[p[x]][p[y]] = m
    return R

def perms(n=5): return [(0,) + q for q in itertools.permutations(range(1, n))]

def enc(T, n=5):                 # the integer key of the paper (5 bits per cell, row-major, first cell = least significant)
    k = 0; sh = 0
    for i in range(1, n):
        for j in range(1, n):
            k |= T[i][j] << sh; sh += 5
    return k

def canon(T, n=5):               # canonical key = minimum over the (n-1)! renamings
    return min(enc(relabel(T, p, n), n) for p in perms(n))

def aut(T, n=5): return [p for p in perms(n) if relabel(T, p, n) == T]

def closure_o(T, A, n=5):        # smallest subset containing A and closed under the hyperoperation
    S = set(A); ch = True
    while ch:
        ch = False
        for x in list(S):
            for y in list(S):
                for z in range(n):
                    if T[x][y] >> z & 1 and z not in S: S.add(z); ch = True
    return S

def base_size(auts, n=5):        # lambda(P): smallest A in P\{e} such that only the identity automorphism fixes A pointwise
    ident = tuple(range(n))
    for r in range(0, n):
        for A in itertools.combinations(range(1, n), r):
            if all(any(p[a] != a for a in A) for p in auts if p != ident): return r

def subpolygroups(T, inv, n=5):
    out = []
    for mask in range(1, 1 << (n - 1)):
        S = {0} | {i + 1 for i in range(n - 1) if mask >> i & 1}
        Sm = sum(1 << s for s in S)
        if all(inv[x] in S for x in S) and all(T[x][y] & ~Sm == 0 for x in S for y in S): out.append(S)
    return out

def fundamental_group_type(T, n=5):    # beta* classes, then the group structure of P/beta*
    allsets = {1 << x for x in range(n)}; fr = set(allsets)
    while fr:
        new = set()
        for S in fr:
            for z in range(n):
                m = 0
                for s in range(n):
                    if S >> s & 1: m |= T[s][z]
                if m not in allsets: new.add(m)
        allsets |= new; fr = new
    parent = list(range(n))
    def f(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for S in allsets:
        el = [i for i in range(n) if S >> i & 1]
        for a in el[1:]: parent[f(a)] = f(el[0])
    cls = {}
    for i in range(n): cls.setdefault(f(i), []).append(i)
    reps = list(cls.values()); k = len(reps)
    idx = {x: ci for ci, c in enumerate(reps) for x in c}
    def mul(a, b):
        m = T[reps[a][0]][reps[b][0]]; s = {idx[z] for z in range(n) if m >> z & 1}
        assert len(s) == 1; return s.pop()
    e = idx[0]
    def order(a):
        cur, o = a, 1
        while cur != e: cur = mul(cur, a); o += 1
        return o
    ords = tuple(sorted(order(a) for a in range(k)))
    return {(1, (1,)): "Z1", (2, (1, 2)): "Z2", (3, (1, 3, 3)): "Z3", (4, (1, 2, 2, 2)): "Klein",
            (4, (1, 2, 4, 4)): "Z4", (5, (1, 5, 5, 5, 5)): "Z5"}[(k, ords)]

def show(T, n=5): return [["{" + ",".join(str(i + 1) for i in range(n) if T[x][y] >> i & 1) + "}" for y in range(n)] for x in range(n)]
print("tools loaded")

tools loaded


## Step 1 — the CSV file of the 4,412 polygroups

In [3]:
tabs = parse_csv(os.path.join(ROOT, "data", "polygroups_order5.csv"))
print("tables read:", len(tabs))
assert len(tabs) == 4412
assert all(is_polygroup(T) for T in tabs)
print("every table satisfies all polygroup axioms (identity, involutive inverse, reversibility, associativity)")
keys = [canon(T) for T in tabs]
assert len(set(keys)) == 4412
print("pairwise non-isomorphic: 4412 distinct canonical keys")
assert all(enc(T) == k for T, k in zip(tabs, keys))
print("every stored table is its own canonical representative")

tables read: 4412


every table satisfies all polygroup axioms (identity, involutive inverse, reversibility, associativity)


pairwise non-isomorphic: 4412 distinct canonical keys
every stored table is its own canonical representative


## Step 2 — independent exhaustive enumeration (orbits of the reversibility relation)

`src/enum_indep.cpp` uses no backtracking and no pruning: for each inverse involution $\rho$ it enumerates all unions of orbits of the triples $(x,y,z)$, $z\in x\circ y$, under $(x,y,z)\mapsto(z,\rho(y),x)$ and $(x,y,z)\mapsto(\rho(x),z,y)$ and keeps those tables with non-empty hyperproducts and associativity.

In [4]:
src = os.path.join(ROOT, "src", "enum_indep.cpp")
exe = os.path.abspath(os.path.join(ROOT, "enum_indep"))
subprocess.run(["g++", "-O2", "-std=c++17", "-o", exe, src], check=True)
raw = {}
for n in (2, 3, 4, 5):
    t0 = time.time()
    r = subprocess.run([exe, str(n)], capture_output=True, text=True, check=True)
    raw[n] = r.stdout.splitlines()
    print(r.stderr.strip().replace("\n", "\n   "), f"   [{time.time()-t0:.1f}s]")

n=2 inverse type with 0 transposition(s): 1 free orbits, 2 labelled solutions    [0.0s]
n=3 inverse type with 0 transposition(s): 4 free orbits, 12 labelled solutions
   n=3 inverse type with 1 transposition(s): 2 free orbits, 3 labelled solutions    [0.0s]
n=4 inverse type with 0 transposition(s): 10 free orbits, 294 labelled solutions
   n=4 inverse type with 1 transposition(s): 7 free orbits, 42 labelled solutions    [0.0s]


n=5 inverse type with 0 transposition(s): 20 free orbits, 63673 labelled solutions
   n=5 inverse type with 1 transposition(s): 16 free orbits, 3720 labelled solutions
   n=5 inverse type with 2 transposition(s): 12 free orbits, 241 labelled solutions    [1.0s]


In [5]:
def classes_of(lines, n):
    out = set(); per_type = collections.Counter()
    for line in lines:
        f = line.split(); per_type[int(f[1])] += 1
        v = list(map(int, f[2:])); T = [v[i * n:(i + 1) * n] for i in range(n)]
        out.add(canon(T, n))
    return out, per_type

indep = {}
for n in (2, 3, 4, 5):
    indep[n], per_type = classes_of(raw[n], n)
    print(f"order {n}: {len(indep[n])} isomorphism classes   (labelled tables per inverse type: {dict(per_type)})")
assert [len(indep[n]) for n in (2, 3, 4)] == [2, 10, 102]
assert len(indep[5]) == 4412
assert indep[5] == set(keys), "the two enumerations differ!"
print("\nThe independent enumeration gives exactly the 4412 classes of the CSV file.")

per_type5 = classes_of(raw[5], 5)[1]
n_norm = per_type5[0] + 6 * per_type5[1] + 3 * per_type5[2]
print("normalised tables of order 5 = ", per_type5[0], "+ 6 *", per_type5[1], "+ 3 *", per_type5[2], "=", n_norm)
assert (per_type5[0], per_type5[1], per_type5[2]) == (63673, 3720, 241) and n_norm == 86716

order 2: 2 isomorphism classes   (labelled tables per inverse type: {0: 2})
order 3: 10 isomorphism classes   (labelled tables per inverse type: {0: 12, 1: 3})
order 4: 102 isomorphism classes   (labelled tables per inverse type: {0: 294, 1: 42})


order 5: 4412 isomorphism classes   (labelled tables per inverse type: {0: 63673, 1: 3720, 2: 241})

The independent enumeration gives exactly the 4412 classes of the CSV file.


normalised tables of order 5 =  63673 + 6 * 3720 + 3 * 241 = 86716


In [6]:
# orbit-stabiliser check: the sum of 24/|Aut(P)| over all classes equals the number of normalised tables
auts = [aut(T) for T in tabs]
total = sum(24 // len(a) for a in auts)
print("sum of 24/|Aut(P)| =", total)
assert total == n_norm == 86716
print("|Aut(P)| distribution:", sorted(collections.Counter(len(a) for a in auts).items()))

sum of 24/|Aut(P)| = 86716
|Aut(P)| distribution: [(1, 2942), (2, 1232), (3, 26), (4, 144), (6, 58), (8, 5), (24, 5)]


## Step 3 — invariants and the tables of the paper

In [7]:
invs, rows = [], []
for T, au in zip(tabs, auts):
    inv = inverse_map(T)
    fixed = sum(1 for x in range(1, 5) if inv[x] == x)
    itype = {4: "1_1_1_1", 2: "2_1_1", 0: "2_2"}[fixed]
    subs = [S for S in subpolygroups(T, inv) if 1 < len(S) < 5]
    gens = [a for a in range(1, 5) if len(closure_o(T, {a})) == 5]
    d = 1 if gens else (2 if any(len(closure_o(T, A)) == 5 for A in itertools.combinations(range(1, 5), 2)) else 3)
    lam = base_size(au)
    comm = all(T[x][y] == T[y][x] for x in range(5) for y in range(5))
    rows.append(dict(index=len(rows) + 1, key=enc(T), inverse_type=itype, primitive=int(not subs), proper_subpolygroup_orders="-".join(map(str, sorted({len(S) for S in subs}))) or "none",
                     commutative=int(comm), fundamental_group=fundamental_group_type(T), alg_gen_number_d=d, num_generating_elements_g=len(gens),
                     aut_order=len(au), logical_gen_number_lambda=lam))
import pandas as pd
df = pd.DataFrame(rows)
print(df.shape)
df.head()

(4412, 11)


,index,key,inverse_type,primitive,proper_subpolygroup_orders,commutative,fundamental_group,alg_gen_number_d,num_generating_elements_g,aut_order,logical_gen_number_lambda
0,1,40296985770984779038849,1_1_1_1,0,2-4,1,Z2,1,1,6,2
1,2,40296985770984913256577,1_1_1_1,0,2-4,1,Z1,1,1,6,2
2,3,40296988031580685754499,1_1_1_1,0,2-4,1,Z2,1,1,1,0
3,4,40296988031580819972227,1_1_1_1,0,2-4,1,Z1,1,1,1,0
4,5,40296990490094798655813,1_1_1_1,0,2-3,1,Z2,1,2,2,1


In [8]:
def pct(c, tot=4412): return f"{100*c/tot:.2f}%"
print("TABLE 2 (Phase counts)\n  non-primitive:", int((df.primitive == 0).sum()), "  primitive:", int(df.primitive.sum()))
print("\nTABLE 4 (inverse types)")
print(pd.crosstab(df.inverse_type, df.primitive.map({0: "non-primitive", 1: "primitive"}), margins=True))
print("\nTABLE 5 (fundamental group)"); print(df.fundamental_group.value_counts().to_string())
print("\ncommutative:", int(df.commutative.sum()), " non-commutative:", int((1 - df.commutative).sum()))
print("classical groups (all products singletons):", sum(all(bin(T[x][y]).count("1") == 1 for x in range(5) for y in range(5)) for T in tabs))
print("\nproper subpolygroup orders:"); print(df.proper_subpolygroup_orders.value_counts().to_string())
print("\nTABLE 6 (algebraic generation)")
print(" d(P):", df.alg_gen_number_d.value_counts().sort_index().to_dict())
print(" g(P):", df.num_generating_elements_g.value_counts().sort_index().to_dict(), [pct(c) for c in df.num_generating_elements_g.value_counts().sort_index()])
print("\nTABLE 7 (logical generation number)")
print(" lambda(P):", df.logical_gen_number_lambda.value_counts().sort_index().to_dict(), [pct(c) for c in df.logical_gen_number_lambda.value_counts().sort_index()])
print("\nTABLE 8 (joint distribution)")
print(pd.crosstab(df.alg_gen_number_d, df.logical_gen_number_lambda, margins=True))

TABLE 2 (Phase counts)
  non-primitive: 1019   primitive: 3393

TABLE 4 (inverse types)
primitive     non-primitive  primitive   All
inverse_type                                
1_1_1_1                 660       2353  3013
2_1_1                   341        975  1316
2_2                      18         65    83
All                    1019       3393  4412

TABLE 5 (fundamental group)
fundamental_group
Z1       4261
Z2        136
Z3         10
Z4          2
Klein       2
Z5          1

commutative: 3776  non-commutative: 636
classical groups (all products singletons): 1

proper subpolygroup orders:
proper_subpolygroup_orders
none     3393
2         619
4         106
3         102
2-3        94
2-4        58
3-4        24
2-3-4      16

TABLE 6 (algebraic generation)
 d(P): {1: 4391, 2: 21}
 g(P): {0: 21, 1: 216, 2: 250, 3: 580, 4: 3345} ['0.48%', '4.90%', '5.67%', '13.15%', '75.82%']

TABLE 7 (logical generation number)
 lambda(P): {0: 2942, 1: 1265, 2: 200, 3: 5} ['66.68%', '28.67%', '

In [9]:
# --- assertions: the numbers printed in the paper ---
assert (df.primitive == 0).sum() == 1019 and df.primitive.sum() == 3393
ct = pd.crosstab(df.inverse_type, df.primitive)
assert ct.loc["1_1_1_1"].tolist() == [660, 2353] and ct.loc["2_1_1"].tolist() == [341, 975] and ct.loc["2_2"].tolist() == [18, 65]
assert df.fundamental_group.value_counts().to_dict() == {"Z1": 4261, "Z2": 136, "Z3": 10, "Klein": 2, "Z4": 2, "Z5": 1}
assert df.commutative.sum() == 3776
assert df.alg_gen_number_d.value_counts().to_dict() == {1: 4391, 2: 21}
assert df.num_generating_elements_g.value_counts().sort_index().tolist() == [21, 216, 250, 580, 3345]
assert df.logical_gen_number_lambda.value_counts().sort_index().tolist() == [2942, 1265, 200, 5]
assert pd.crosstab(df.alg_gen_number_d, df.logical_gen_number_lambda).values.tolist() == [[2941, 1254, 192, 4], [1, 11, 8, 1]]
assert df.logical_gen_number_lambda.max() <= 3          # Corollary: lambda(P) <= |P|-2
assert (df.logical_gen_number_lambda == 0).sum() == (df.aut_order == 1).sum()   # lambda = 0  iff  Aut(P) = 1
assert ((df.logical_gen_number_lambda == 3) == (df.aut_order == 24)).all()
print("all numbers of the paper are reproduced")

all numbers of the paper are reproduced


### Examples of the paper (Section 5)

In [10]:
# Example 1: x o y = {x,y} (x != y), x o x = P \ {x}  -- cyclic, Aut = S4, lambda = 3
K = [[0] * 5 for _ in range(5)]
for x in range(5): K[0][x] = K[x][0] = 1 << x
for x in range(1, 5):
    for y in range(1, 5): K[x][y] = (1 << x) | (1 << y) if x != y else 31 & ~(1 << x)
assert is_polygroup(K) and len(aut(K)) == 24 and base_size(aut(K)) == 3 and len(closure_o(K, {1})) == 5
print("Example 1: polygroup, cyclic, |Aut| = 24, lambda = 3;  in the list:", canon(K) in set(keys))

# Example 2: the unique non-cyclic rigid polygroup
rigid = [T for T, a, r in zip(tabs, auts, rows) if len(a) == 1 and r["alg_gen_number_d"] == 2]
assert len(rigid) == 1
for r in show(rigid[0]): print("   ", r)
print("Example 2: unique polygroup with d(P) = 2 and lambda(P) = 0")

Example 1: polygroup, cyclic, |Aut| = 24, lambda = 3;  in the list: True
    ['{1}', '{2}', '{3}', '{4}', '{5}']
    ['{2}', '{2}', '{2,3}', '{2,3,4}', '{1,2,5}']
    ['{3}', '{3}', '{2,3}', '{1,2,3,4,5}', '{3,4,5}']
    ['{4}', '{3,4}', '{1,2,3,4,5}', '{4,5}', '{4,5}']
    ['{5}', '{1,2,3,4,5}', '{3,4}', '{4}', '{5}']
Example 2: unique polygroup with d(P) = 2 and lambda(P) = 0


## Step 4 (optional) — rerun the main two-phase search

Set `RUN_MAIN_SEARCH = True` to recompile and run `src/polygroups_n5.cpp` (about 15 min on 4 cores, 40 min on 2 cores). The program prints the Phase 1 / Phase 2 statistics and re-verifies its own output; the notebook then checks that the produced list equals `data/polygroups_order5.csv`.

In [11]:
RUN_MAIN_SEARCH = False
if RUN_MAIN_SEARCH:
    work = os.path.abspath("search_run"); os.makedirs(work, exist_ok=True)
    exe2 = os.path.join(work, "polygroups_n5")
    subprocess.run(["g++", "-O2", "-std=c++17", "-fopenmp", "-o", exe2, os.path.abspath(os.path.join(ROOT, "src", "polygroups_n5.cpp"))], check=True)
    t0 = time.time()
    p = subprocess.run([exe2], cwd=work, capture_output=True, text=True)
    print(p.stdout.replace("\r", "\n")[-2500:]); print(f"run time: {(time.time()-t0)/60:.1f} min")
    new = parse_csv(os.path.join(work, "polygroups_final_results.csv"))
    assert len(new) == 4412 and {canon(T) for T in new} == set(keys)
    print("the search program reproduces exactly the 4412 classes")
else:
    print("skipped (set RUN_MAIN_SEARCH = True to run it)")

skipped (set RUN_MAIN_SEARCH = True to run it)


In [12]:
# write the per-polygroup invariant table (this file is also stored in data/)
out = os.path.join(ROOT, "data", "polygroups_order5_invariants.csv")
df.to_csv(out, index=False)
print("written:", out)
print("\nALL CHECKS PASSED")

written: ../data/polygroups_order5_invariants.csv

ALL CHECKS PASSED
